In [ ]:
import json
import os

import mlflow
from datasets import load_dataset
from dotenv import load_dotenv
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from mlflow.genai.datasets import create_dataset, delete_dataset
from mlflow.genai.scorers import Safety

load_dotenv()

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "dataset_tracking_example"

d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Warning - This step may take a while to download the dataset - The dataset size is ~40GB
wikipedia_dataset = load_dataset("wikimedia/wikipedia", "20231101.en")
wikipedia_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'url', 'title', 'text'],
        num_rows: 6407814
    })
})

In [3]:
KEYWORD = "artificial intelligence"


# Filter for articles where the specific keyword if it appears at least 3 times
def filter_article_by_keyword(article):
    KEYWORD = "artificial intelligence"
    text_lower = article["text"].lower()
    count = text_lower.count(KEYWORD)
    return count >= 3


filtered_articles_dataset = wikipedia_dataset["train"].filter(
    filter_article_by_keyword,
    desc="Filtering articles with at least 3 occurrences of the keyword",
    batch_size=1000,
    writer_batch_size=1000,
    num_proc=4,
)

filtered_articles_dataset

Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 1683
})

In [ ]:
filtered_articles_df = filtered_articles_dataset.to_pandas()
filtered_articles_id_title_json = filtered_articles_df[["id", "title"]].to_json(
    orient="records"
)
MODEL = "openai"

if MODEL == "google":
    rate_limiter = InMemoryRateLimiter(
        requests_per_second=0.1,  # <-- Super slow! We can only make a request once every 10 seconds!!
        check_every_n_seconds=0.1,  # Wake up every 100 ms to check whether allowed to make a request,
        max_bucket_size=10,  # Controls the maximum burst size.
    )

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash-lite",
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
        rate_limiter=rate_limiter,
    )
elif MODEL == "openai":
    llm = ChatOpenAI(
        model="gpt-5-nano",
    )
else:
    raise ValueError(f"Unsupported MODEL: {MODEL}")

prompt = f"""Your are provided with the id and title of the wikipedia articles. Identify the top 100 articles which matches best to the Topic - {KEYWORD} and return the ids of the articles in a list format.
Make sure to return only 100 ids in a json list format.
{filtered_articles_id_title_json}
"""

predicted_ids = llm.invoke(prompt)
predicted_ids


AIMessage(content='["713","1164","1208","2142","2846","2862","4715","5561","5626","4103607","4244428","4253446","4420730","4559681","4678739","4852787","5033373","5071866","5175523","5254736","5439872","5732209","5837233","5841092","5843242","5906926","6320384","6338699","6341170","6422674","6476399","6477222","6585513","6610123","6691035","6961226","7092991","7093162","7101688","7249174","7398678","7406885","7408685","7644182","7755275","8233911","8383635","8434470","8532924","8734826","8919856","9034035","11701105","12592050","13230426","13494976","13587263","13710305","13744516","14027739","14659967","14705543","14844609","14949242","15710171","15795950","15800444","15874732","16070200","16167377","16300571","16713196","17114897","17332858","17651946","17667553","17953841","18365276","18402617","18686460","19024298","19301286","19667111","19768790","20269843","20756850","20903754","20914512","21109528","21313292","21410069","21476402","21548766","21659435","21666977","26086272","262

In [13]:
predicted_ids_list = json.loads(
    predicted_ids.content.replace("```json", "").replace("```", "")
)
len(predicted_ids_list)

100

In [14]:
filtered_articles_df = filtered_articles_df[
    filtered_articles_df["id"].isin(predicted_ids_list)
]
print(filtered_articles_df.shape)
filtered_articles_df.head(10)

(100, 4)


,id,url,title,text
0,713,https://en.wikipedia.org/wiki/Android%20%28rob...,Android (robot),An android is a humanoid robot or other artifi...
1,1164,https://en.wikipedia.org/wiki/Artificial%20int...,Artificial intelligence,Artificial intelligence (AI) is the intelligen...
2,1208,https://en.wikipedia.org/wiki/Alan%20Turing,Alan Turing,Alan Mathison Turing (; 23 June 1912 – 7 June...
4,2142,https://en.wikipedia.org/wiki/List%20of%20arti...,List of artificial intelligence projects,"The following is a list of current and past, n..."
5,2846,https://en.wikipedia.org/wiki/Ai,Ai,"AI is artificial intelligence, intellectual ab..."
6,2862,https://en.wikipedia.org/wiki/AI-complete,AI-complete,"In the field of artificial intelligence, the m..."
7,4715,https://en.wikipedia.org/wiki/Boolean%20satisf...,Boolean satisfiability problem,"In logic and computer science, the Boolean sat..."
9,5561,https://en.wikipedia.org/wiki/Computational%20...,Computational linguistics,Computational linguistics is an interdisciplin...
10,5626,https://en.wikipedia.org/wiki/Cognitive%20science,Cognitive science,"Cognitive science is the interdisciplinary, sc..."
14,4103607,https://en.wikipedia.org/wiki/Dalle%20Molle%20...,Dalle Molle Institute for Artificial Intellige...,The Dalle Molle Institute for Artificial Intel...


In [15]:
dataset = mlflow.data.from_pandas(
    df=filtered_articles_df, name=f"top_100_articles_{KEYWORD.replace(' ', '_')}"
)

with mlflow.start_run():
    mlflow.log_input(dataset, context="raw_data")

🏃 View run likeable-pig-924 at: http://localhost:5000/#/experiments/1/runs/51adb853f82245579328534d6920ccce
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [16]:
data_list = []

for record in filtered_articles_df["text"].tolist():
    data_list.append({"inputs": {"text": record}, "expectations": {}})
data_list[0]

{'inputs': {'text': 'An android is a humanoid robot or other artificial being often made from a flesh-like material. Historically, androids were completely within the domain of science fiction and frequently seen in film and television, but advances in robot technology now allow the design of functional and realistic humanoid robots.\n\nTerminology\n\nThe Oxford English Dictionary traces the earliest use (as "Androides") to Ephraim Chambers\' 1728 Cyclopaedia, in reference to an automaton that St. Albertus Magnus allegedly created. By the late 1700s, "androides", elaborate mechanical devices resembling humans performing human activities, were displayed in exhibit halls.\nThe term "android" appears in US patents as early as 1863 in reference to miniature human-like toy automatons. The term android was used in a more modern sense by the French author Auguste Villiers de l\'Isle-Adam in his work Tomorrow\'s Eve (1886). This story features an artificial humanlike robot named Hadaly. As sai

In [17]:
summary_dataset = create_dataset(
    name="summarization_dataset",
    tags={"type": "summary", "source": "wikipedia"},
)

In [18]:
summary_dataset.merge_records(data_list[:10])

In [ ]:
# from mlflow.genai.datasets import get_dataset
# get_dataset(dataset_id=summary_dataset.dataset_id)

In [19]:
def predict_fn(text) -> str:
    prompt = f"""Summarize the text below as a bullet point list of the most important points.
    Text: {text}"""
    response = llm.invoke(prompt)
    return response.content


# 3.Run the evaluation
results = mlflow.genai.evaluate(
    data=summary_dataset, predict_fn=predict_fn, scorers=[Safety()]
)

2026/01/28 22:49:05 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/01/28 22:49:05 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 10/10 [Elapsed: 01:03, Remaining: 00:00] 


In [21]:
results.result_df

,trace_id,safety/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-d9f9425788716db6de5510e015044688,yes,"{""info"": {""trace_id"": ""tr-d9f9425788716db6de55...",None,OK,1769620779360,11549,{'text': 'The Dalle Molle Institute for Artifi...,"- IDSIA is a research institute in Lugano, Can...","{'mlflow.traceOutputs': '""- IDSIA is a researc...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': '2flCV4hxbbbeVRDgFQRGiA==', 'spa...",[{'assessment_id': 'a-772cd3af32a84073bccca958...
1,tr-ba518c423a87703a5c096f8f3d7da88a,yes,"{""info"": {""trace_id"": ""tr-ba518c423a87703a5c09...",None,OK,1769620779360,21340,{'text': 'In the field of artificial intellige...,- AI-complete (AI-hard) problems are as diffic...,"{'mlflow.traceOutputs': '""- AI-complete (AI-ha...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'ulGMQjqHcDpcCW+PPX2oig==', 'spa...",[{'assessment_id': 'a-6eb8a2e8f3bd480388f07ba7...
2,tr-3c703ab3072e248397b2510c40a14bd0,yes,"{""info"": {""trace_id"": ""tr-3c703ab3072e248397b2...",None,OK,1769620779370,30231,{'text': 'Alan Mathison Turing (; 23 June 191...,- Alan Turing (1912–1954) was an English mathe...,"{'mlflow.traceOutputs': '""- Alan Turing (1912–...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'PHA6swcuJIOXslEMQKFL0A==', 'spa...",[{'assessment_id': 'a-9be405a5c873490dac31f30e...
3,tr-7bd930ef60ae25fe7940ef1d8b8c1482,yes,"{""info"": {""trace_id"": ""tr-7bd930ef60ae25fe7940...",None,OK,1769620779357,31406,{'text': 'Cognitive science is the interdiscip...,- Cognitive science is an interdisciplinary st...,"{'mlflow.traceOutputs': '""- Cognitive science ...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'e9kw72CuJf55QO8di4wUgg==', 'spa...",[{'assessment_id': 'a-9042cd3c34884c40a32d2724...
4,tr-677f73263a0396752a91c907ca1e9619,yes,"{""info"": {""trace_id"": ""tr-677f73263a0396752a91...",None,OK,1769620779368,31037,"{'text': 'In logic and computer science, the B...",- SAT (Boolean satisfiability problem) asks wh...,"{'mlflow.traceOutputs': '""- SAT (Boolean satis...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'Z39zJjoDlnUqkckHyh6WGQ==', 'spa...",[{'assessment_id': 'a-3546de2fc0484431b61a5394...
5,tr-5704c46a8a875b1dd5d1d12b81fe83e2,yes,"{""info"": {""trace_id"": ""tr-5704c46a8a875b1dd5d1...",None,OK,1769620779326,34310,{'text': 'An android is a humanoid robot or ot...,- Androids are humanoid robots or artificial b...,"{'mlflow.traceOutputs': '""- Androids are human...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'VwTEaoqHWx3V0dErgf6D4g==', 'spa...",[{'assessment_id': 'a-a9047cf22f8146fbb89cfd86...
6,tr-8f0cab499331010fc98fb7590cdd6782,yes,"{""info"": {""trace_id"": ""tr-8f0cab499331010fc98f...",None,OK,1769620779353,42314,{'text': 'Computational linguistics is an inte...,- Computational linguistics is an interdiscipl...,"{'mlflow.traceOutputs': '""- Computational ling...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'jwyrSZMxAQ/Jj7dZDN1ngg==', 'spa...",[{'assessment_id': 'a-4674fb19349645628603c49b...
7,tr-eb07b45f48ebc67e0c40503017a3219b,yes,"{""info"": {""trace_id"": ""tr-eb07b45f48ebc67e0c40...",None,OK,1769620779360,44289,{'text': 'The following is a list of current a...,- The text is a catalog of current and past no...,"{'mlflow.traceOutputs': '""- The text is a cata...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': '6we0X0jrxn4MQFAwF6Mhmw==', 'spa...",[{'assessment_id': 'a-9767a00e6c6644bab6890352...
8,tr-f28bdebc9cb56cbda06f70d5652e91d6,yes,"{""info"": {""trace_id"": ""tr-f28bdebc9cb56cbda06f...",None,OK,1769620779344,49125,"{'text': 'AI is artificial intelligence, intel...",- AI typically stands for artificial intellige...,"{'mlflow.traceOutputs': '""- AI typically stand...",{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': '8ovevJy1bL2gb3DVZS6R1g==', 'spa...",[{'assessment_id'

In [22]:
annotated_dataset = create_dataset(
    name="annotated_dataset",
    tags={"type": "annotated", "source": "wikipedia"},
)

In [24]:
annotated_data_list = []
for index, row in results.result_df[["request", "response"]].iterrows():
    annotated_data_list.append(
        {
            "inputs": {"text": row["request"]["text"]},
            "expectations": {"response": row["response"]},
        }
    )

annotated_data_list[0]

{'inputs': {'text': 'The Dalle Molle Institute for Artificial Intelligence Research (, IDSIA) is a research institution based in Lugano, in  Canton Ticino in southern Switzerland. It was founded in 1988 by Angelo Dalle Molle through the private Fondation Dalle Molle. In 2000 it became a public research institute, affiliated with the Università della Svizzera italiana and SUPSI in Ticino, Switzerland. In 1997 it was listed among the top ten artificial intelligence laboratories, and among the top four in the field of biologically-inspired AI.\n\nIn 2007 a robotics lab with focus on intelligent and learning robots, especially in the fields of swarm and humanoid robotics, was established.\n\nBetween 2009 and 2012, artificial neural networks developed at the institute won eight international competitions in pattern recognition and machine learning.\n\nIDSIA is one of four Swiss research organisations founded by the Dalle Molle foundation, of which three are in the field of artificial intell

In [25]:
annotated_dataset.merge_records(annotated_data_list)

In [26]:
delete_dataset(dataset_id=annotated_dataset.dataset_id)

In [27]:
traces = mlflow.search_traces(max_results=10, return_type="list", run_id=results.run_id)
traces

[Trace(trace_id=tr-3c703ab3072e248397b2510c40a14bd0),
 Trace(trace_id=tr-677f73263a0396752a91c907ca1e9619),
 Trace(trace_id=tr-ba518c423a87703a5c096f8f3d7da88a),
 Trace(trace_id=tr-d9f9425788716db6de5510e015044688),
 Trace(trace_id=tr-eb07b45f48ebc67e0c40503017a3219b),
 Trace(trace_id=tr-7bd930ef60ae25fe7940ef1d8b8c1482),
 Trace(trace_id=tr-8f0cab499331010fc98fb7590cdd6782),
 Trace(trace_id=tr-f28bdebc9cb56cbda06f70d5652e91d6),
 Trace(trace_id=tr-c993569370a9148722a649f80e10aec6),
 Trace(trace_id=tr-5704c46a8a875b1dd5d1d12b81fe83e2)]

In [28]:
annotated_data_list = []
for trace in traces:
    trace_id = trace.to_dict()["info"]["trace_id"]
    request = trace.data._get_root_span().inputs["text"]
    response = trace.data._get_root_span().outputs
    annotated_data_list.append(
        {"inputs": {"text": request}, "expectations": {"response": response}}
    )

annotated_data_list[0]

{'inputs': {'text': 'Alan Mathison Turing  (; 23 June 1912\xa0– 7 June 1954) was an English mathematician, computer scientist, logician, cryptanalyst, philosopher and theoretical biologist. Turing was highly influential in the development of theoretical computer science, providing a formalisation of the concepts of algorithm and computation with the Turing machine, which can be considered a model of a general-purpose computer. He is widely considered to be the father of theoretical computer science and artificial intelligence.\n\nBorn in Maida Vale, London, Turing was raised in southern England. He graduated at King\'s College, Cambridge, with a degree in mathematics. Whilst he was a fellow at Cambridge, he published a proof demonstrating that some purely mathematical yes–no questions can never be answered by computation. He defined a Turing machine and proved that the halting problem for Turing machines is undecidable. In 1938, he obtained his PhD from the Department of Mathematics at

In [29]:
annotated_dataset_from_trace = create_dataset(
    name="annotated_dataset_from_trace",
    tags={"type": "annotated", "source": "wikipedia"},
)

In [30]:
annotated_dataset_from_trace.merge_records(annotated_data_list)